In [ ]:
import torch
import timm

print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)
print("GPU available:", torch.cuda.is_available())

## Imports
All library imports used throughout the notebook.

In [ ]:

import pandas as pd
import numpy as np
import os
from collections import Counter
from tqdm import tqdm
import timm

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


## Dataset Paths
Set up data directory and verify the metadata CSV exists.

In [ ]:
import os

DATA_DIR      = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
METADATA_PATH = os.path.join(DATA_DIR, "HAM10000_metadata.csv")

if os.path.exists(METADATA_PATH):
    print(f"✓ Metadata found: {METADATA_PATH}")
else:
    print(f"✗ ERROR: Not found at {METADATA_PATH}")

## Basic Information

In [ ]:
df = pd.read_csv(METADATA_PATH)
print(f"--Total images: {len(df)}")
print(f"--Unique lesions: {df['lesion_id'].nunique()}")
print(f"--Classes: {df['dx'].nunique()}")
display(df.head())

print("\n--Datatypes:--\n")
print(df.dtypes)

print(f"\n--Missing Values:--")
print(df.isnull().sum())

print(f"\n--Dataset Shape:--")
print(df.shape)

df.describe()

## Missing value handling


In [ ]:

df['age'].fillna(df['age'].mean(), inplace=True)
print(df.isnull().sum())

## Class distribution analysis

In [ ]:
# Get class counts
class_counts = df['dx'].value_counts()

# Class names (full descriptions)
class_names = {
    'nv': 'Melanocytic nevi',
    'mel': 'Melanoma',
    'bkl': 'Benign keratosis',
    'bcc': 'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc': 'Vascular lesions',
    'df': 'Dermatofibroma'
}

print("\n--Class Distribution:--")

for cls in class_counts.index:
    count = class_counts[cls]
    percentage = (count / len(df)) * 100
    print(f"{cls:6s} -> {class_names[cls]:25s}: {count:5d} ({percentage:5.2f}%)")

print("\n--Imbalance Ratio:--")
max_class = class_counts.max()
min_class = class_counts.min()
imbalance_ratio = max_class / min_class
print(f"  Most common class: {class_counts.index[0]} ({max_class} images)")
print(f"  Least common class: {class_counts.index[-1]} ({min_class} images)")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")

## Visualize class distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(14, 6))

colors = sns.color_palette("husl", len(class_counts))
bars = ax.bar(
    range(len(class_counts)),
    class_counts.values,
    color=colors,
    edgecolor='black',
    linewidth=1.5
)

ax.set_xlabel('Diagnosis Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Images', fontsize=13, fontweight='bold')
ax.set_title('Class Distribution - Showing Imbalance Problem', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels(class_counts.index, fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels
for bar, count in zip(bars, class_counts):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f'{count} ({count/len(df)*100:.1f}%)',
        ha='center', va='bottom'
    )

plt.show()


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))

ax.pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',
    startangle=90
)

ax.set_title('Class Distribution (HAM10000)', fontsize=14, fontweight='bold')

plt.show()


## Data leakage analysis

In [ ]:

# Count how many images each lesion has
images_per_lesion = df.groupby('lesion_id').size()

print(f"\n--Basic Statistics:--")
print(f"  - Total unique lesions: {len(images_per_lesion):,}")
print(f"  - Total images: {len(df):,}")
print(f"  - Mean images per lesion: {images_per_lesion.mean():.2f}")
print(f"  - Max images per lesion: {images_per_lesion.max()}")
print(f"  - Min images per lesion: {images_per_lesion.min()}")

# Distribution breakdown
print(f"\n--Distribution Breakdown:--")
single_image = (images_per_lesion == 1).sum()
multiple_images = (images_per_lesion > 1).sum()

single_image_percentage = single_image/len(images_per_lesion)*100
multiple_images_percentage = multiple_images/len(images_per_lesion)*100

print(f"  - Lesions with only 1 image: {single_image:,} ({single_image_percentage:.1f}%)")
print(f"  - Lesions with 2+ images: {multiple_images:,} ({multiple_images_percentage:.1f}%)")


print("\n--Quantifying The Data Leakage Risk:--")

total_images = len(df)
images_from_multi_lesions = total_images - single_image

print(f"  - Total images: {total_images:,}")
print(f"  - Images from lesions with 2+ photos: {images_from_multi_lesions:,}")
print(f"  - Percentage at risk: {images_from_multi_lesions/total_images*100:.1f}%")
print(f"\nThis means {images_from_multi_lesions/total_images*100:.1f}% of the dataset is at risk")


## Image Resizing
Define function to resize all HAM10000 images to 224×224 and save to disk.

In [ ]:
from tqdm import tqdm

def resize_and_save_images(df):
    """
    Resize all images and save to disk.
    
    Args:
        df : DataFrame with 'image_id' column
    
    Returns:
        missing : List of image_ids that were not found
    """
    IMAGE_SIZE = (224, 224)
    IMAGE_DIRS = [
        '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
        '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/ham10000_images_part_1',
        '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2',
        '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/ham10000_images_part_2',
    ]
    OUTPUT_DIR = '/kaggle/working/archive/HAM10000_resized'

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    saved   = 0
    missing = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Resizing"):
        out_path = os.path.join(OUTPUT_DIR, row['image_id'] + '.jpg')

        if os.path.exists(out_path):
            saved += 1
            continue

        for img_dir in IMAGE_DIRS:
            img_path = os.path.join(img_dir, row['image_id'] + '.jpg')
            if os.path.exists(img_path):
                img = Image.open(img_path).convert('RGB')
                img = img.resize(IMAGE_SIZE)
                img.save(out_path, quality=95)
                saved += 1
                break
        else:
            missing.append(row['image_id'])

    print(f"\n--Summary--")
    print(f"  - Saved:   {saved:,}")
    print(f"  - Missing: {len(missing):,}")

    return missing

## Dataset Statistics
Compute per-channel mean and std of the resized dataset for normalization.

In [ ]:
import torch
from tqdm import tqdm
from torchvision import transforms

print("--Computing HAM10000 Mean & Std--")

def compute_mean_std(df):
    """
    Compute per-channel mean and std of resized images.
    
    Args:
        df : DataFrame with 'image_id' column
    
    Returns:
        mean : Tensor of shape [3]
        std  : Tensor of shape [3]
    """
    OUTPUT_DIR = '/kaggle/working/archive/HAM10000_resized'

    to_tensor = transforms.Compose([
        transforms.ToTensor()
    ])

    channel_sum    = torch.zeros(3)
    channel_sq_sum = torch.zeros(3)
    count = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Computing stats"):
        img_path = os.path.join(OUTPUT_DIR, row['image_id'] + '.jpg')
        img    = Image.open(img_path).convert('RGB')
        tensor = to_tensor(img)

        channel_sum    += tensor.sum(dim=[1, 2])
        channel_sq_sum += (tensor ** 2).sum(dim=[1, 2])
        count          += tensor.shape[1] * tensor.shape[2]

    mean = channel_sum / count
    std  = (channel_sq_sum / count - mean ** 2) ** 0.5

    print(f"\n--HAM10000 Dataset Statistics--")
    print(f"  - Mean: {mean.tolist()}")
    print(f"  - Std:  {std.tolist()}")

    return mean, std
   

## Random Dataset Split
Random train/val/test split (no lesion ID leakage check).

In [ ]:
def split_dataset_random(df):
    """
    Args:
        df : DataFrame with 'image_id' column
    
    Returns:
        train_df : Training DataFrame
        val_df   : Validation DataFrame
        test_df  : Test DataFrame
    """
    print("--Splitting Dataset Randomly (No Lesion ID Check)--")

    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
    val_df,  test_df  = train_test_split(temp_df, test_size=0.50, random_state=42)

    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    print(f"\n--Split Summary--")
    print(f"  - Train images: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"  - Val images:   {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
    print(f"  - Test images:  {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")
    print(f"  - Total:        {len(train_df)+len(val_df)+len(test_df):,}")

    return train_df, val_df, test_df

## Stratified Dataset Split 
Split by unique lesion IDs to prevent data leakage between splits.

In [ ]:
def split_dataset(df):
    """
    Args:
        df : DataFrame with 'lesion_id' and 'image_id' columns
    
    Returns:
        train_df : Training DataFrame
        val_df   : Validation DataFrame
        test_df  : Test DataFrame
    """
    print("--Splitting Dataset by Lesion ID--")

    unique_lesions = df['lesion_id'].unique()
    print(f"\n  - Total unique lesions: {len(unique_lesions):,}")

    train_lesions, temp_lesions = train_test_split(unique_lesions,
                                                    test_size=0.30,
                                                    random_state=42)
    val_lesions, test_lesions   = train_test_split(temp_lesions,
                                                    test_size=0.50,
                                                    random_state=42)

    train_df = df[df['lesion_id'].isin(train_lesions)].reset_index(drop=True)
    val_df   = df[df['lesion_id'].isin(val_lesions)].reset_index(drop=True)
    test_df  = df[df['lesion_id'].isin(test_lesions)].reset_index(drop=True)

    print(f"\n--Split Summary--")
    print(f"  - Train images: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"  - Val images:   {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
    print(f"  - Test images:  {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")
    print(f"  - Total:        {len(train_df)+len(val_df)+len(test_df):,}")

    # ── Leakage Check ──────────────────────────────────────────────────────────
    print(f"\n--Leakage Check--")
    train_lesion_set = set(train_df['lesion_id'])
    val_lesion_set   = set(val_df['lesion_id'])
    test_lesion_set  = set(test_df['lesion_id'])

    train_val_overlap  = train_lesion_set & val_lesion_set
    train_test_overlap = train_lesion_set & test_lesion_set
    val_test_overlap   = val_lesion_set   & test_lesion_set

    print(f"  - Train/Val overlap:  {len(train_val_overlap)}")
    print(f"  - Train/Test overlap: {len(train_test_overlap)}")
    print(f"  - Val/Test overlap:   {len(val_test_overlap)}")

    if len(train_val_overlap) == len(train_test_overlap) == len(val_test_overlap) == 0:
        print(f"\n  ✓ No leakage detected! Split is clean.")
    else:
        print(f"\n  ✗ Leakage detected! Something went wrong.")

    return train_df, val_df, test_df

## Custom Transforms
Define custom augmentation classes (e.g. sigmoid intensity correction).

In [ ]:

# ── Sigmoid Intensity Correction (custom, not in torchvision) ─────────────────
class SigmoidIntensityCorrection:
    def __init__(self, gain=10, cutoff=0.5):
        self.gain = gain          # empirically tuned
        self.cutoff = cutoff

    def __call__(self, img):
        img_array = np.array(img).astype(np.float32) / 255.0
        corrected = 1 / (1 + np.exp(-self.gain * (img_array - self.cutoff)))
        corrected = (corrected * 255).astype(np.uint8)
        return Image.fromarray(corrected)


## Transform Pipeline
Build train and val/test transform pipelines using augmentation flags.

In [ ]:
import torchvision.transforms.functional as TF
import random
import math

def get_transforms(mean, std, use_augmentation=True):
    """
    Get train and val/test transforms.

    Args:
        mean             : Channel mean tensor from compute_mean_std()
        std              : Channel std tensor from compute_mean_std()
        use_augmentation : If True, apply augmentation to train transform

    Returns:
        train_transform     : Transform for training set
        val_test_transform  : Transform for val and test sets
    """
    IMAGE_SIZE = (224, 224)

    if use_augmentation:
        train_transform = transforms.Compose([
            transforms.RandomRotation(degrees=30),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomResizedCrop(size=IMAGE_SIZE, scale=(0.80, 1.0)),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2)),
            transforms.RandomAffine(degrees=0, shear=(-0.2, 0.2), scale=(0.8, 1.2)),
            SigmoidIntensityCorrection(gain=10, cutoff=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean.tolist(), std=std.tolist())
        ])
    else:
        train_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=mean.tolist(), std=std.tolist())
        ])

    val_test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean.tolist(), std=std.tolist())
    ])

    return train_transform, val_test_transform

## Oversampling
Oversample minority classes and optionally downsample the majority class to balance the dataset.

In [ ]:
from collections import Counter

def oversample_full_dataset(df, target_per_class=6705):
    """
    Oversample minority classes + downsample majority class
    to target_per_class. Runs on FULL df BEFORE any split.
    """
    print(f"\nOversampling to {target_per_class} per class...")
    class_counts = Counter(df['dx'])

    oversampled = []
    for cls in class_counts:
        cls_df = df[df['dx'] == cls]
        if len(cls_df) < target_per_class:
            # Minority class — oversample
            extra = cls_df.sample(
                n=target_per_class - len(cls_df),
                replace=True, random_state=42
            )
            cls_df = pd.concat([cls_df, extra])
        else:
            # Majority class (NV=6705) — keep as is
            cls_df = cls_df.sample(
                n=target_per_class, random_state=42
            )
        oversampled.append(cls_df)

    balanced_df = pd.concat(oversampled).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)

    print(f"Balanced dataset: {len(balanced_df):,} images")
    for cls, cnt in sorted(Counter(balanced_df['dx']).items()):
        print(f"  {cls}: {cnt:,}")
    print(f"Expected: {target_per_class * 7:,}  ({target_per_class} × 7 classes)")
    return balanced_df

## Dataset Class
`HAMDataset` — PyTorch Dataset wrapping image loading, label mapping, and transforms.

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

label_map = {
    'nv'   : 0,
    'mel'  : 1,
    'bkl'  : 2,
    'bcc'  : 3,
    'akiec': 4,
    'df'   : 5,
    'vasc' : 6
}

class HAMDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe
        self.img_dir   = '/kaggle/working/archive/HAM10000_resized'
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_id'] + '.jpg')
        img      = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = label_map[row['dx']]
        return img, label


def get_dataloaders(train_df, val_df, test_df, train_transform, val_test_transform, batch_size=32):
    """
    Create datasets and dataloaders.

    Args:
        train_df            : Training DataFrame (already oversampled if needed)
        val_df              : Validation DataFrame
        test_df             : Test DataFrame
        train_transform     : Transform for training set
        val_test_transform  : Transform for val and test sets
        batch_size          : Batch size, default 32

    Returns:
        train_loader : Training DataLoader
        val_loader   : Validation DataLoader
        test_loader  : Test DataLoader
    """
    train_dataset = HAMDataset(train_df, transform=train_transform)
    val_dataset   = HAMDataset(val_df,   transform=val_test_transform)
    test_dataset  = HAMDataset(test_df,  transform=val_test_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2, drop_last=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

    print("--Dataset Summary--")
    print(f"  - Train samples: {len(train_dataset):,}")
    print(f"  - Val samples:   {len(val_dataset):,}")
    print(f"  - Test samples:  {len(test_dataset):,}")

    print("\n--Dataloader Summary--")
    print(f"  - Train batches: {len(train_loader):,}")
    print(f"  - Val batches:   {len(val_loader):,}")
    print(f"  - Test batches:  {len(test_loader):,}")

    print("\n--Verifying 1 batch--")
    images, labels = next(iter(train_loader))
    print(f"  - Image batch shape: {images.shape}")
    print(f"  - Label batch shape: {labels.shape}")
    print(f"  - Label samples:     {labels[:5].tolist()}")
    print(f"  - Min pixel value:   {images.min():.4f}")
    print(f"  - Max pixel value:   {images.max():.4f}")

    return train_loader, val_loader, test_loader

## Training Loop
Single-epoch training function with loss and accuracy tracking.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
 
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
 
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
 
    return total_loss / total, correct / total

In [ ]:
import timm

def get_model(model_name='levit'):
    if model_name == 'levit':
        model = timm.create_model('levit_256', pretrained=True, num_classes=7)
    elif model_name == 'resnet50':
        model = timm.create_model('resnet50', pretrained=True, num_classes=7)
    elif model_name == 'xception':
        model = timm.create_model('xception', pretrained=True, num_classes=7)
    return model.to(DEVICE)

## Full Training Pipeline
Two-phase training: frozen backbone (head-only) → full fine-tune with LR scheduling.

In [ ]:
def train_model(train_loader, val_loader, num_epochs=30, lr=1e-4, criterion=None):
    """
    Full training loop:
      - Phase 1: Freeze backbone, train head only (5 epochs)
      - Phase 2: Unfreeze all, fine-tune end-to-end (remaining epochs)
      - Optimizer: AdamW with weight_decay=0.01 (paper Table 4)
    Returns model + history dict for learning curve plotting.
    """
    model     = get_model()
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [],
               'val_loss':   [], 'val_acc':  [], 'val_f1': []}

    # Phase 1: Freeze backbone, train head only
    print("\n  [Phase 1] Training classifier head only...")
    for param in model.parameters():
        param.requires_grad = False
    for param in model.head.parameters():
        param.requires_grad = True

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3, weight_decay=0.01
    )

    for epoch in range(5):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_f1, val_spec, val_mcc, val_pr_auc, _, _, _ = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(f"  Epoch [{epoch+1}/5]  "
              f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
              f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

    # Phase 2: Unfreeze all, fine-tune full model
    print("\n  [Phase 2] Fine-tuning full model...")
    for param in model.parameters():
        param.requires_grad = True

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - 5)

    best_val_f1  = 0
    best_weights = None

    for epoch in range(num_epochs - 5):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_f1, val_spec, val_mcc, val_pr_auc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(f"  Epoch [{epoch+1}/{num_epochs-5}]  "
              f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
              f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}  "
              f"Val F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            print(f"    ✓ Best model saved (Val F1: {best_val_f1:.4f})")

    model.load_state_dict(best_weights)
    return model, history

## Reproducibility
Set all random seeds for deterministic runs.

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:

missing = resize_and_save_images(df)
print(f"Resize done. Missing: {len(missing)}")

## K-Fold Cross Validation — Session 1 Setup
Imports, constants, and data loading for the 2-fold CV experiment.

In [ ]:
 
import os, json, time, shutil
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, matthews_corrcoef,
    average_precision_score, confusion_matrix,
    classification_report
)
from sklearn.preprocessing import label_binarize
 
# ── Constants (same in both sessions) ────────────────────────────────────────
METADATA_PATH    = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
CKPT_DIR         = '/kaggle/working/checkpoints'
N_SPLITS         = 2       # K=2
NUM_EPOCHS       = 20      # epoch=20
TARGET_PER_CLASS = 3000    # per class images
LR               = 1e-4
 
MEAN = [0.7634057402610779,  0.5461452007293701,  0.570540189743042]
STD  = [0.14088796079158783, 0.15225186944007874, 0.16917429864406586]
 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Device : {DEVICE}")
print(f"K      : {N_SPLITS}")
print(f"Epochs : {NUM_EPOCHS}")
print(f"Per class: {TARGET_PER_CLASS}")
 
# ── Load df ───────────────────────────────────────────────────────────────────
df = pd.read_csv(METADATA_PATH)
df['age'] = df['age'].fillna(df['age'].mean())
print(f"\nRaw dataset: {len(df):,} images")
 
# ── Resize (skip if already done) ────────────────────────────────────────────
missing = resize_and_save_images(df)
print(f"Resize done. Missing: {len(missing)}")
 
# ── Oversample FULL dataset FIRST — paper Section 3.3 ────────────────────────

balanced_df = oversample_full_dataset(df, target_per_class=TARGET_PER_CLASS)
 

## Evaluate Function
Compute all four paper metrics: Accuracy, F1, Specificity, MCC, PR-AUC.

In [ ]:
 
def evaluate(model, loader, criterion, n_classes=7):
    """All 4 paper metrics: F1, Specificity, MCC, PR AUC."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            probs   = torch.softmax(outputs, dim=1)
 
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)
 
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
 
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
 
    avg_loss = total_loss / total
    accuracy = correct / total
    f1       = f1_score(all_labels, all_preds, average='weighted')
    mcc      = matthews_corrcoef(all_labels, all_preds)
 
    labels_bin = label_binarize(all_labels, classes=list(range(n_classes)))
    pr_auc     = average_precision_score(labels_bin, all_probs, average='weighted')
 
    cm            = confusion_matrix(all_labels, all_preds, labels=list(range(n_classes)))
    class_support = np.bincount(all_labels, minlength=n_classes)
    specs = []
    for i in range(n_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    specificity = np.average(specs, weights=class_support)
 
    return (avg_loss, accuracy, f1, specificity, mcc, pr_auc,
            all_preds.tolist(), all_labels.tolist(), all_probs.tolist())
 

In [ ]:
 
def run_folds(balanced_df, mean, std, fold_start, fold_end):
    label_names  = ['NV', 'MEL', 'BKL', 'BCC', 'AKIEC', 'DF', 'VASC']
    results_path = os.path.join(CKPT_DIR, 'results.csv')
    criterion    = nn.CrossEntropyLoss()
 
    train_transform, val_test_transform = get_transforms(
        mean, std, use_augmentation=True
    )
 
    skf              = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=42
    )
    labels_for_split = balanced_df['dx'].values
 
    # ── Resume check ──────────────────────────────────────────────────────────
    if os.path.exists(results_path):
        fold_results = pd.read_csv(results_path).to_dict('records')
        done_folds   = {r['Fold'] for r in fold_results}
        print(f"Already completed folds: {sorted(done_folds)}")
    else:
        fold_results = []
        done_folds   = set()
 
    for fold, (train_idx, val_idx) in enumerate(
            skf.split(balanced_df, labels_for_split)):
 
        fold_num = fold + 1
 
        if fold_num < fold_start or fold_num > fold_end:
            continue
        if fold_num in done_folds:
            print(f"Fold {fold_num} already done — skipping.")
            continue
 
        print(f"\n{'='*60}")
        print(f"  FOLD {fold_num} / {N_SPLITS}")
        print(f"  Train: {len(train_idx):,} | Val: {len(val_idx):,}")
        print(f"{'='*60}")
        set_seed(42 + fold)
 
        train_df = balanced_df.iloc[train_idx].reset_index(drop=True)
        val_df   = balanced_df.iloc[val_idx].reset_index(drop=True)
 
        # Sanity check
        val_counts = Counter(val_df['dx'])
        expected   = len(val_idx) // 7
        print(f"Val per class (expected ~{expected}):")
        for cls in sorted(val_counts):
            print(f"  {cls}: {val_counts[cls]}")
 
        train_loader, val_loader, _ = get_dataloaders(
            train_df, val_df, val_df,
            train_transform, val_test_transform, batch_size=32
        )
 
        # Train
        model, history = train_model(
            train_loader, val_loader,
            num_epochs=NUM_EPOCHS, lr=LR,
            criterion=criterion
        )
 
        # Evaluate
        _, acc, f1, spec, mcc, pr_auc, preds, labels, _ = evaluate(
            model, val_loader, criterion
        )
 
        print(f"\n── Fold {fold_num} Results ──")
        print(f"  F1 Score    : {f1*100:.2f}%   (paper: 96.11%)")
        print(f"  Specificity : {spec*100:.2f}%  (paper: 96.29%)")
        print(f"  MCC         : {mcc*100:.2f}%   (paper: 95.51%)")
        print(f"  PR AUC      : {pr_auc*100:.2f}% (paper: 96.62%)")
        print(f"  Accuracy    : {acc*100:.2f}%")
 
        fold_results.append({
            'Fold'       : fold_num,
            'Accuracy'   : round(acc,    4),
            'F1'         : round(f1,     4),
            'Specificity': round(spec,   4),
            'MCC'        : round(mcc,    4),
            'PR_AUC'     : round(pr_auc, 4),
        })
 
        # Save checkpoint
        torch.save(model.state_dict(),
                   os.path.join(CKPT_DIR, f'fold_{fold_num}_model.pth'))
        with open(os.path.join(CKPT_DIR,
                               f'fold_{fold_num}_history.json'), 'w') as fh:
            json.dump(history, fh)
        np.savez(os.path.join(CKPT_DIR, f'fold_{fold_num}_preds.npz'),
                 preds=np.array(preds), labels=np.array(labels))
        pd.DataFrame(fold_results).to_csv(results_path, index=False)
        print(f"✓ Checkpoint saved → {CKPT_DIR}/fold_{fold_num}_*")
 
    # Summary
    results_df = pd.DataFrame(fold_results)
    print(f"\n{'='*50}")
    print(f"Completed folds: {sorted(results_df['Fold'].tolist())}")
    for metric in ['F1', 'Specificity', 'MCC', 'PR_AUC']:
        m = results_df[metric].mean() * 100
        s = results_df[metric].std()  * 100
        print(f"  {metric:<14}: {m:.2f}% ± {s:.2f}%")
    print(f"{'='*50}")
 
    return results_df
 

In [ ]:

mean = torch.tensor(MEAN)
std  = torch.tensor(STD)
 
results_df = run_folds(
    balanced_df, mean, std,
    fold_start=1,
    fold_end=1       
)
 
# Files to download
print("\n── Download these files from Output panel ──")
for f in sorted(os.listdir(CKPT_DIR)):
    size = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1024 / 1024
    print(f"  {f:<40}  {size:.1f} MB")
 


In [ ]:
import os
for item in os.listdir('/kaggle/input/datasets/sadiatabassumsadia'):
    print(item)

In [ ]:

#  SESSION 2 CELL 1 — Load Fold 1 checkpoint

 
PREV_CKPT = '/kaggle/input/datasets/sadiatabassumsadia/levit-ckpt-fold1'  

CKPT_DIR  = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
 
# আগের fold 1 checkpoint copy করো
for fname in os.listdir(PREV_CKPT):
    shutil.copy2(
        os.path.join(PREV_CKPT, fname),
        os.path.join(CKPT_DIR,  fname)
    )
    print(f"Copied: {fname}")
 
# Verify
prev_results = pd.read_csv(os.path.join(CKPT_DIR, 'results.csv'))
print(f"\nFold 1 results loaded:")
print(prev_results.to_string(index=False))
 

In [ ]:

# SESSION 2 CELL 2 — Run Fold 2


df          = pd.read_csv(METADATA_PATH)
df['age']   = df['age'].fillna(df['age'].mean())
balanced_df = oversample_full_dataset(df, target_per_class=TARGET_PER_CLASS)
 
mean = torch.tensor(MEAN)
std  = torch.tensor(STD)
 
results_df = run_folds(
    balanced_df, mean, std,
    fold_start=2,
    fold_end=2       
)
 

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/fold2_checkpoints', 'zip', '/kaggle/working/checkpoints')
print("Done!")

In [ ]:
import os, json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ── Paths ────────────────────────────────────────────────────
FOLD2_DIR = '/kaggle/input/datasets/sadiatabassumsadia/levit-ckpt-fold2'
FOLD1_DIR = '/kaggle/input/datasets/sadiatabassumsadia/levit-ckpt-fold1'
OUT_DIR   = '/kaggle/working'

# ── Load histories ───────────────────────────────────────────
with open(os.path.join(FOLD1_DIR, 'fold_1_history.json')) as f:
    h1 = json.load(f)
with open(os.path.join(FOLD2_DIR, 'fold_2_history.json')) as f:
    h2 = json.load(f)

# ── Load preds ───────────────────────────────────────────────
d1 = np.load(os.path.join(FOLD1_DIR, 'fold_1_preds.npz'))
d2 = np.load(os.path.join(FOLD2_DIR, 'fold_2_preds.npz'))

all_preds  = np.concatenate([d1['preds'],  d2['preds']])
all_labels = np.concatenate([d1['labels'], d2['labels']])

label_names = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']

# ════════════════════════════════════════════════════════════
# 1. Confusion Matrix
# ════════════════════════════════════════════════════════════
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_title('LEViT Confusion Matrix — 2-Fold CV', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print(classification_report(all_labels, all_preds, target_names=label_names))

# ════════════════════════════════════════════════════════════
# 2. Learning Curves — Fold 1 + Fold 2 side by side
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LEViT Learning Curves (Fold 1 vs Fold 2)', fontsize=14, fontweight='bold')

for fold_idx, (h, label) in enumerate([(h1, 'Fold 1'), (h2, 'Fold 2')]):
    epochs = range(1, len(h['train_loss']) + 1)

    # Loss
    axes[fold_idx][0].plot(epochs, h['train_loss'], label='Train', color='royalblue', lw=2)
    axes[fold_idx][0].plot(epochs, h['val_loss'],   label='Val',   color='tomato',    lw=2)
    axes[fold_idx][0].axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='Phase 2')
    axes[fold_idx][0].set_title(f'{label} — Loss')
    axes[fold_idx][0].set_xlabel('Epoch')
    axes[fold_idx][0].legend()
    axes[fold_idx][0].grid(alpha=0.3)

    # Accuracy
    axes[fold_idx][1].plot(epochs, h['train_acc'], label='Train', color='royalblue', lw=2)
    axes[fold_idx][1].plot(epochs, h['val_acc'],   label='Val',   color='tomato',    lw=2)
    axes[fold_idx][1].axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='Phase 2')
    axes[fold_idx][1].set_title(f'{label} — Accuracy')
    axes[fold_idx][1].set_xlabel('Epoch')
    axes[fold_idx][1].legend()
    axes[fold_idx][1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'learning_curves.png'), dpi=150)
plt.show()
print('✓ Saved: confusion_matrix.png, learning_curves.png')

In [ ]:
import pandas as pd


results_df = pd.read_csv('/kaggle/input/datasets/sadiatabassumsadia/levit-ckpt-fold2/results.csv')

# Mean ± Std row
mean_row = {'Fold': 'Mean'}
std_row  = {'Fold': 'Std'}
for m in ['Accuracy', 'F1', 'Specificity', 'MCC', 'PR_AUC']:
    mean_row[m] = round(results_df[m].mean(), 4)
    std_row[m]  = round(results_df[m].std(),  4)

paper_row = {'Fold': 'Paper', 'Accuracy': '-', 'F1': 0.9611, 'Specificity': 0.9629, 'MCC': 0.9551, 'PR_AUC': 0.9662}

final_df = pd.concat([results_df, pd.DataFrame([mean_row, std_row, paper_row])], ignore_index=True)
final_df.to_csv('/kaggle/working/final_results.csv', index=False)
print(final_df.to_string(index=False))
print('\n✓ Saved: final_results.csv')

In [ ]:
import torch
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model = timm.create_model('levit_256', pretrained=False, num_classes=7)
model.load_state_dict(torch.load('/kaggle/input/datasets/sadiatabassumsadia/levit-ckpt-fold1/fold_1_model.pth', 
                                  map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print("✓ Model loaded successfully")
print(f"Device: {DEVICE}")


for name, module in model.named_modules():
    print(name)

## Grad-CAM Visualization
Visualize EigenCAM activations for one sample per class using the best fold checkpoint.

In [ ]:
!pip install grad-cam -q

import torch
import timm
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
label_names = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
MEAN = [0.7634, 0.5461, 0.5705]
STD  = [0.1409, 0.1523, 0.1692]

# ── Model load ───────────────────────────────────────────────
model = timm.create_model('levit_256', pretrained=False, num_classes=7)
model.load_state_dict(torch.load(
    '/kaggle/input/datasets/sadiatabassumsadia/levit-ckpt-fold1/fold_1_model.pth',
    map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# ── Target layer ─────────────────────────────────────────────
target_layer = [model.stem.conv4.linear]
# ── Transform ────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

IMAGE_DIRS = [
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2',
]

import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')


samples = df.groupby('dx').first().reset_index()[['dx', 'image_id']]

# ── Grad-CAM plot ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 7, figsize=(20, 9))
fig.suptitle('LEViT Grad-CAM Visualization — HAM10000', fontsize=14, fontweight='bold')

from pytorch_grad_cam import EigenCAM


cam = EigenCAM(model=model, target_layers=target_layer)

for col, (_, row) in enumerate(samples.iterrows()):
    img_path = None
    for d in IMAGE_DIRS:
        p = os.path.join(d, row['image_id'] + '.jpg')
        if os.path.exists(p):
            img_path = p
            break
    if img_path is None:
        continue

    # Original image
    orig = Image.open(img_path).convert('RGB').resize((224, 224))
    orig_np = np.array(orig) / 255.0

    # Tensor
    input_tensor = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)

    # Prediction
    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = output.argmax(1).item()

    # Grad-CAM
    grayscale_cam = cam(input_tensor=input_tensor)[0]
    cam_image = show_cam_on_image(orig_np.astype(np.float32), grayscale_cam, use_rgb=True)

    # Plot
    axes[0][col].imshow(orig)
    axes[0][col].set_title(f'{label_names[col]}', fontsize=10, fontweight='bold')
    axes[0][col].axis('off')

    axes[1][col].imshow(cam_image)
    axes[1][col].set_title(f'Pred: {label_names[pred_idx]}', fontsize=9,
                            color='green' if pred_idx == col else 'red')
    axes[1][col].axis('off')

    # Heatmap only
    heatmap = cv2.applyColorMap(np.uint8(255 * grayscale_cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    axes[2][col].imshow(heatmap)
    axes[2][col].set_title('Heatmap', fontsize=9)
    axes[2][col].axis('off')

axes[0][0].set_ylabel('Original', fontsize=10)
axes[1][0].set_ylabel('Grad-CAM', fontsize=10)
axes[2][0].set_ylabel('Heatmap', fontsize=10)

plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gradcam_visualization.png')